# Nautiq - setup del entorno

Configuración centralizada para el flujo activo del TFM:

- Acceso seguro a Bronze mediante SAS Token.
- Rutas de los topics AIS.
- Estado técnico de Auto Loader.
- Tablas Silver DEV.
- Gold histórico de port calls.
- Gold de baseline histórico de espera por eslora y por tipo de buque.
- Gold de predicciones JIT actuales.
- Modelo ML registrado en Unity Catalog con alias `Champion`.
- Cambio futuro entre tablas administradas y ADLS externo.

### Notebooks activos

- `silver_ais_positions_dev`
- `silver_ais_static_dev`
- `gold_vessel_port_calls_jit`
- `gold_waiting_avg_per_length`
- `ml_vessel_jit_classification`
- `gold_vessel_jit_current_predictions`


In [0]:
# ============================================================
# CONFIGURACION GENERAL DEL PROYECTO
# ============================================================

project_name = "nautiq"
environment = "dev"
bronze_start_date = "2026-08-01"

# Puertos analizados en el proyecto
ports_data = [
    ("ESVLC", "Valencia", 39.430018, -0.309838),
    ("ESBCN", "Barcelona", 41.312166, 2.209423),
    ("ESALG", "Algeciras", 36.166941, -5.412677),
]

# Catalogo usado durante las pruebas en Databricks
catalog_name = "masterxyz002dbr"

# Schemas de Unity Catalog
silver_schema = "silver"
gold_schema = "gold"
ops_schema = "ops"

# Volume tecnico para schemas y checkpoints de Auto Loader
ops_volume_name = f"{project_name}_{environment}"

# Zona horaria comun para todos los procesos
spark.conf.set("spark.sql.session.timeZone", "UTC")


# ============================================================
# CONFIGURACION DEL ADLS DE ORIGEN
# ============================================================
source_storage_account = "stnautiqdatadevswc"
bronze_container = "bronze"

bronze_secret_scope = "nautiq-secrets"
bronze_secret_key = "nautiq-sas-token"

# ============================================================
# CONFIGURACION DEL DESTINO
# ============================================================
# Valores permitidos:
# managed  -> tablas administradas en tu Unity Catalog
# external -> tablas Delta almacenadas en el ADLS del proyecto
target_storage_mode = "managed"
# Durante las pruebas no se utiliza para escribir.
# Se deja preparado para el futuro cambio a ADLS.
target_storage_account = "stnautiqdatadevswc"

silver_container = "silver"
gold_container = "gold"

# Ruta interna dentro de los contenedores Silver y Gold
external_project_path = f"{project_name}/{environment}"

# Secrets futuros para escribir en Silver y Gold.
# No se utilizan mientras target_storage_mode sea managed.
silver_secret_scope = "nautiq-secrets"
silver_secret_key = "nautiq-silver-sas-token"

gold_secret_scope = "nautiq-secrets"
gold_secret_key = "nautiq-gold-sas-token"

In [0]:
# ============================================================
# FUNCION DE CONFIGURACION SAS
# ============================================================

def configure_wasbs_sas(
    storage_account: str,
    container_name: str,
    secret_scope: str,
    secret_key: str
) -> None:
    """
    Configura el acceso WASBS a un contenedor de Azure Storage
    utilizando un SAS Token almacenado en Databricks Secrets.
    """

    sas_token = (
        dbutils.secrets
        .get(scope=secret_scope, key=secret_key)
        .strip()
        .lstrip("?")
    )

    if not sas_token:
        raise ValueError(
            f"El SAS Token del contenedor {container_name} esta vacio."
        )

    spark_conf_key = (
        f"fs.azure.sas.{container_name}."
        f"{storage_account}.blob.core.windows.net"
    )

    spark.conf.set(spark_conf_key, sas_token)


# Configura solamente el acceso de lectura a Bronze
configure_wasbs_sas(
    storage_account=source_storage_account,
    container_name=bronze_container,
    secret_scope=bronze_secret_scope,
    secret_key=bronze_secret_key
)
# En modo external se configura también el acceso
# a los contenedores Silver y Gold del ADLS.
if target_storage_mode == "external":

    configure_wasbs_sas(
        storage_account=target_storage_account,
        container_name=silver_container,
        secret_scope=silver_secret_scope,
        secret_key=silver_secret_key
    )

    configure_wasbs_sas(
        storage_account=target_storage_account,
        container_name=gold_container,
        secret_scope=gold_secret_scope,
        secret_key=gold_secret_key
    )

In [0]:
# ============================================================
# CREACION DE OBJETOS DE UNITY CATALOG
# ============================================================

schemas = [
    silver_schema,
    gold_schema,
    ops_schema
]

for schema_name in schemas:
    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        `{catalog_name}`.`{schema_name}`
        """
    )


spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS
    `{catalog_name}`.`{ops_schema}`.`{ops_volume_name}`
    """
)

DataFrame[]

In [0]:
# ============================================================
# RUTAS BRONZE
# ============================================================

bronze_root = (
    f"wasbs://{bronze_container}@{source_storage_account}"
    f".blob.core.windows.net/dev/telemetry"
)

positions_source = f"{bronze_root}/positions/"
static_source = f"{bronze_root}/static/"

In [0]:
# ============================================================
# ESTADO TECNICO DE AUTO LOADER
# ============================================================
ops_root = (
    f"/Volumes/{catalog_name}/"
    f"{ops_schema}/{ops_volume_name}"
)

# Dataset: ais_position_reports
positions_schema_path = (
    f"{ops_root}/schemas/ais_position_reports_dev_telemetry"
)

positions_checkpoint_path = (
    f"{ops_root}/checkpoints/ais_position_reports_dev_telemetry"
)

# Dataset: ais_static_voyage_reports
static_schema_path = (
    f"{ops_root}/schemas/ais_static_voyage_reports_dev_telemetry"
)

static_checkpoint_path = (
    f"{ops_root}/checkpoints/ais_static_voyage_reports_dev_telemetry"
)

In [0]:
# ============================================================
# TABLAS SILVER DEV
# ============================================================

positions_target_table = f"{catalog_name}.{silver_schema}.ais_positions_dev"
static_target_table = f"{catalog_name}.{silver_schema}.ais_static_dev"


# ============================================================
# TABLAS GOLD ACTIVAS
# ============================================================

vessel_port_calls_target_table = f"{catalog_name}.{gold_schema}.vessel_port_calls_analytics"
waiting_avg_per_length_target_table = f"{catalog_name}.{gold_schema}.waiting_avg_per_length"
waiting_avg_per_type_target_table = f"{catalog_name}.{gold_schema}.waiting_avg_per_type"
vessel_jit_predictions_target_table = f"{catalog_name}.{gold_schema}.vessel_jit_current_predictions"


# ============================================================
# MODELO ML REGISTRADO
# ============================================================

registered_model_name = f"{catalog_name}.{gold_schema}.vessel_jit_classifier"
registered_model_alias = "Champion"
registered_model_uri = f"models:/{registered_model_name}@{registered_model_alias}"


# ============================================================
# NOTEBOOKS ACTIVOS DEL FLUJO
# ============================================================

positions_silver_notebook = "silver_ais_positions_dev"
static_silver_notebook = "silver_ais_static_dev"
port_calls_gold_notebook = "gold_vessel_port_calls_jit"
waiting_gold_notebook = "gold_waiting_avg_per_length"
ml_notebook = "ml_vessel_jit_classification"
prediction_gold_notebook = "gold_vessel_jit_current_predictions"


In [0]:
# ============================================================
# RUTAS EXTERNAS PARA EL ADLS DEL PROYECTO
# ============================================================

silver_external_root = f"wasbs://{silver_container}@{target_storage_account}.blob.core.windows.net/{external_project_path}"
gold_external_root = f"wasbs://{gold_container}@{target_storage_account}.blob.core.windows.net/{external_project_path}"

positions_external_path = f"{silver_external_root}/ais_positions_dev"
static_external_path = f"{silver_external_root}/ais_static_dev"

vessel_port_calls_external_path = f"{gold_external_root}/vessel_port_calls_analytics"
waiting_avg_per_length_external_path = f"{gold_external_root}/waiting_avg_per_length"
waiting_avg_per_type_external_path = f"{gold_external_root}/waiting_avg_per_type"
vessel_jit_predictions_external_path = f"{gold_external_root}/vessel_jit_current_predictions"


In [0]:
# ============================================================
# RESOLUCION DEL DESTINO DE ESCRITURA
# ============================================================

def get_target_path(external_path: str) -> str | None:
    if target_storage_mode == "managed":
        return None
    if target_storage_mode == "external":
        return external_path
    raise ValueError("target_storage_mode debe ser managed o external.")

positions_target_path = get_target_path(positions_external_path)
static_target_path = get_target_path(static_external_path)

vessel_port_calls_target_path = get_target_path(vessel_port_calls_external_path)
waiting_avg_per_length_target_path = get_target_path(waiting_avg_per_length_external_path)
waiting_avg_per_type_target_path = get_target_path(waiting_avg_per_type_external_path)
vessel_jit_predictions_target_path = get_target_path(vessel_jit_predictions_external_path)


In [ ]:
# ============================================================
# ESCRITURA DELTA MANAGED / EXTERNAL
# ============================================================

def write_delta_table(
    df,
    target_table: str,
    target_path: str | None,
    mode: str = "overwrite"
) -> None:

    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if target_path is not None:
        writer = writer.option("path", target_path)

    writer.saveAsTable(target_table)

In [0]:
# ============================================================
# CONFIGURACION DE DATASETS
# ============================================================

datasets = {
    "ais_position_reports": {
        "source_topic": "telemetry.positions",
        "source_path": positions_source,
        "schema_path": positions_schema_path,
        "checkpoint_path": positions_checkpoint_path,
        "target_table": positions_target_table,
        "target_path": positions_target_path
    },
    "ais_static_voyage_reports": {
        "source_topic": "telemetry.static",
        "source_path": static_source,
        "schema_path": static_schema_path,
        "checkpoint_path": static_checkpoint_path,
        "target_table": static_target_table,
        "target_path": static_target_path
    }
}

In [0]:
# ============================================================
# VALIDACION DEL ENTORNO
# ============================================================

if target_storage_mode not in {"managed", "external"}:
    raise ValueError("target_storage_mode debe ser managed o external.")

source_paths = {
    "ais_positions": positions_source,
    "ais_static": static_source
}

print("==============================================")
print("NAUTIQ - CONFIGURACION DEL ENTORNO")
print("==============================================")
print(f"Environment: {environment}")
print(f"Catalog: {catalog_name}")
print(f"Target storage mode: {target_storage_mode}")
print(f"Ops volume: {ops_root}")
print()

for dataset_name, source_path in source_paths.items():
    try:
        source_items = dbutils.fs.ls(source_path)
        print(f"[OK] {dataset_name}: {len(source_items)} elementos encontrados")
    except Exception as error:
        raise RuntimeError(f"No se puede acceder a {dataset_name}: {source_path}") from error

print()
print("Silver DEV tables:")
print(f"  - {positions_target_table}")
print(f"  - {static_target_table}")

print()
print("Gold tables:")
print(f"  - {vessel_port_calls_target_table}")
print(f"  - {waiting_avg_per_length_target_table}")
print(f"  - {waiting_avg_per_type_target_table}")
print(f"  - {vessel_jit_predictions_target_table}")

print()
print("Registered ML model:")
print(f"  - {registered_model_name}@{registered_model_alias}")

print()
print("Active notebooks:")
print(f"  - {positions_silver_notebook}")
print(f"  - {static_silver_notebook}")
print(f"  - {port_calls_gold_notebook}")
print(f"  - {waiting_gold_notebook}")
print(f"  - {ml_notebook}")
print(f"  - {prediction_gold_notebook}")

print()
print("[OK] Setup completado correctamente.")


NAUTIQ - CONFIGURACION DEL ENTORNO
Environment: dev
Catalog: masterxyz002dbr
Target storage mode: managed
Ops volume: /Volumes/masterxyz002dbr/ops/nautiq_dev

[OK] ais_positions: 26 elementos encontrados
[OK] ais_static: 34 elementos encontrados

Silver DEV tables:
  - masterxyz002dbr.silver.ais_positions_dev
  - masterxyz002dbr.silver.ais_static_dev

Gold tables:
  - masterxyz002dbr.gold.vessel_port_calls_analytics
  - masterxyz002dbr.gold.waiting_avg_per_length
  - masterxyz002dbr.gold.waiting_avg_per_type
  - masterxyz002dbr.gold.vessel_jit_current_predictions

Registered ML model:
  - masterxyz002dbr.gold.vessel_jit_classifier@Champion

Active notebooks:
  - silver_ais_positions_dev
  - silver_ais_static_dev
  - gold_vessel_port_calls_jit
  - gold_waiting_avg_per_length
  - ml_vessel_jit_classification
  - gold_vessel_jit_current_predictions

[OK] Setup completado correctamente.
